### Data Ingestion


LOading


In [1]:
import os
import json
import chromadb
from sentence_transformers import SentenceTransformer



c:\Users\Dell\postmate\MYpostman-APP\Backend\RAG-local\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import os
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# 1. Navigate from the notebook folder to the RAG-folder
# Path("../RAG-folder") means: Go up one level, then into RAG-folder
target_path = os.path.join("..", "RAG-folder")

print(f"--- Attempting to load from: {os.path.abspath(target_path)} ---")

# 2. Setup the DirectoryLoader
dlr_loader = DirectoryLoader(
    target_path, 
    glob="*.json",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

# 3. Execution and Verification
try:
    documents = dlr_loader.load()
    
    if not documents:
        print("⚠️  No files found. Verify that RAG-folder contains .json files.")
    else:
        print(f"✅ Success! Loaded {len(documents)} documents.")
        # Print the first 100 characters of the first file to confirm
        print(f"Preview: {documents[0].page_content[:100]}...")

except Exception as e:
    print(f"❌ Error: {e}")

--- Attempting to load from: c:\Users\Dell\postmate\MYpostman-APP\Backend\RAG-local\RAG-folder ---


100%|██████████| 1/1 [00:00<00:00, 25.64it/s]

✅ Success! Loaded 1 documents.
Preview: [
  {
    "id": 1,
    "category": "Technology Announcement",
    "tone": "Professional and concise"...


### Embeddings and Vector DB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import uuid
from chromadb.config import Settings
import chromadb
from typing import List, Dict, Any
import os




In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

class EmbeddingManager:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        """
        Constructor for EmbeddingManager.

        Args:
            model_name (str): SentenceTransformer model name
        """
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name, device=self.device)
            print(f"Model '{self.model_name}' loaded successfully on {self.device}.")
            print(f"Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, text: str | list[str]) -> np.ndarray:
        """
        Generate embeddings for a single text or a list of texts.

        Args:
            text (str | list[str]): Text or list of texts to embed.

        Returns:
            list[float]: Single embedding if text is a string.
            list[list[float]]: List of embeddings if text is a list.
            return array of embeddings as numpy array
        """
        if self.model is None:
            raise RuntimeError("Model is not loaded. Call _load_model() first.")
        embeddings = self.model.encode(text, normalize_embeddings=True)
        print(f"Generated embeddings for input: {text}")
        print(f"Embedding shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
sample_text = "This is a sample text to generate embeddings."

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1943.45it/s]


Model 'sentence-transformers/all-MiniLM-L6-v2' loaded successfully on cpu.
Embedding dimension: 384


### Vector Store

In [15]:
class VectorStore:
    def __init__(self, collection_name: str = "my_collection", persist_directory: str = "vector_store"):
        """
        Constructor for VectorStore.

        Args:
            collection_name (str): Name of the ChromaDB collection.
            persist_directory (str): Directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
   
        self._initialize_vector_store()

    def _initialize_vector_store(self):                          # ✅ level 1
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection for storing document embeddings and metadata"}
            )
            print(f"Vector store initialized with collection '{self.collection_name}' at '{self.persist_directory}'.")
            print(f"Collection metadata: {self.collection.metadata}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):   # ✅ level 1
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must be the same.")

        ids, metadatas, document_texts, embeddings_list = [], [], [], []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            document_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:                                                     # ✅ outside loop
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_texts
            )
            print(f"Added {len(documents)} documents to the vector store.")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

    

In [16]:
vectorstore=VectorStore()
vectorstore

Vector store initialized with collection 'my_collection' at 'vector_store'.
Collection metadata: {'description': 'Collection for storing document embeddings and metadata'}


### Creating Data Chunks

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [18]:
chunk=split_documents(documents)
chunk

Split 1 documents into 17 chunks

Example chunk:
Content: [
  {
    "id": 1,
    "category": "Technology Announcement",
    "tone": "Professional and concise",
    "content": "Just deployed the latest Docker containers for the MYpostmate backend! If anyone i...
Metadata: {'source': '..\\RAG-folder\\top-posts-archive.json'}


[Document(metadata={'source': '..\\RAG-folder\\top-posts-archive.json'}, page_content='[\n  {\n    "id": 1,\n    "category": "Technology Announcement",\n    "tone": "Professional and concise",\n    "content": "Just deployed the latest Docker containers for the MYpostmate backend! If anyone is struggling with port mapping between Node.js and PostgreSQL, let\'s discuss it here. \\ud83d\\ude80 #TechTalk"\n  },\n  {\n    "id": 2,\n    "category": "Community Discussion",\n    "tone": "Casual and helpful",\n    "content": "Friendly reminder: your reputation score directly affects your dashboard visibility. Keep the discussions clean and help out your fellow devs! What\'s everyone working on this weekend?"\n  },\n  {\n    "id": 3,\n    "category": "Debugging Help",\n    "tone": "Technical and analytical",\n    "content": "Stuck on a weird React state rendering issue. The component remounts twice on initial load. I\'ve checked the useEffect dependencies. Any ideas? Snippet attached below."\n  

In [19]:
## convert chunks text into embeddings
texts=[doc.page_content for doc in chunk]
texts

['[\n  {\n    "id": 1,\n    "category": "Technology Announcement",\n    "tone": "Professional and concise",\n    "content": "Just deployed the latest Docker containers for the MYpostmate backend! If anyone is struggling with port mapping between Node.js and PostgreSQL, let\'s discuss it here. \\ud83d\\ude80 #TechTalk"\n  },\n  {\n    "id": 2,\n    "category": "Community Discussion",\n    "tone": "Casual and helpful",\n    "content": "Friendly reminder: your reputation score directly affects your dashboard visibility. Keep the discussions clean and help out your fellow devs! What\'s everyone working on this weekend?"\n  },\n  {\n    "id": 3,\n    "category": "Debugging Help",\n    "tone": "Technical and analytical",\n    "content": "Stuck on a weird React state rendering issue. The component remounts twice on initial load. I\'ve checked the useEffect dependencies. Any ideas? Snippet attached below."\n  },\n  {\n    "id": 4,\n    "category": "Tech Discussion",\n    "tone": "Academic and 

In [20]:

embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunk, embeddings)

Generated embeddings for input: ['[\n  {\n    "id": 1,\n    "category": "Technology Announcement",\n    "tone": "Professional and concise",\n    "content": "Just deployed the latest Docker containers for the MYpostmate backend! If anyone is struggling with port mapping between Node.js and PostgreSQL, let\'s discuss it here. \\ud83d\\ude80 #TechTalk"\n  },\n  {\n    "id": 2,\n    "category": "Community Discussion",\n    "tone": "Casual and helpful",\n    "content": "Friendly reminder: your reputation score directly affects your dashboard visibility. Keep the discussions clean and help out your fellow devs! What\'s everyone working on this weekend?"\n  },\n  {\n    "id": 3,\n    "category": "Debugging Help",\n    "tone": "Technical and analytical",\n    "content": "Stuck on a weird React state rendering issue. The component remounts twice on initial load. I\'ve checked the useEffect dependencies. Any ideas? Snippet attached below."\n  },\n  {\n    "id": 4,\n    "category": "Tech Discussi

### Retrival RAG pipeline fetch from vector DB

In [22]:
from typing import List, Dict, Any

class Retrieval:
    def __init__(self, vectorstore, embedding_manager):
        """ 
        Initializes the Retrieval class with a vector store and an embedding manager. 
        """
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, score_threshold: float = 0.0, top_k: int = 5) -> List[Dict[str, Any]]:
        """ 
        Retrieves relevant documents based on a query.
        """
        print(f"Retrieving documents for query: '{query}' with score threshold: {score_threshold} and top_k: {top_k}")
        
        # Step 1: Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        print(f"Query embedding shape: {query_embedding.shape}")
        
        # Step 2: Retrieve relevant documents from the vector store
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                # ✅ FIX 1: Added "distances" to the include list so we can calculate the score
                include=["documents", "metadatas", "distances", "embeddings"] 
            )
            
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                # ✅ FIX 2: Added the missing comma after 'i' and fixed 'metadeta' spelling
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    
                    # ChromaDB uses Cosine Distance by default (0 is perfect match, 1 is completely different)
                    # Converting distance to a Similarity Score (1 is perfect, 0 is different)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'content': document,
                            'metadata': metadata,
                            'score': similarity_score,
                            'id': doc_id,
                            'rank': i + 1,
                            'distance': distance
                        })
                        # Moved this print statement outside the append block so it doesn't print for every single document
                    else:
                        print(f"Document '{doc_id}' filtered out due to low similarity score: {similarity_score:.4f}")

            print(f"Total retrieved: {len(retrieved_docs)} documents after applying score threshold.")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise
retrieval = Retrieval(vectorstore, embedding_manager)
retrieval.retrieve("What is the content of the document?", score_threshold=0.5, top_k=3)

Retrieving documents for query: 'What is the content of the document?' with score threshold: 0.5 and top_k: 3
Generated embeddings for input: ['What is the content of the document?']
Embedding shape: (1, 384)
Query embedding shape: (384,)
Document 'doc_c4e77dcd_13' filtered out due to low similarity score: -0.6166
Document 'doc_12688127_15' filtered out due to low similarity score: -0.6200
Document 'doc_45240fac_14' filtered out due to low similarity score: -0.6218
Total retrieved: 0 documents after applying score threshold.


[]